# Data Profiling — Federal County-Level Health & Socioeconomic Analytics Pipeline

This notebook profiles each raw federal dataset before cleaning. It documents:
- Structure (rows, columns)
- Data types and missing values
- Key variables and join keys
- Data quality issues


In [ ]:
import pandas as pd
import os

DATA_DIR = "data/raw"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 10)

## 1. ACS Raw Data

Source: U.S. Census Bureau — American Community Survey 5-Year
Purpose: Socioeconomic indicators (population, income, poverty)

In [ ]:
acs = pd.read_csv(os.path.join(DATA_DIR, "acs_raw.csv"))
print(f"Shape: {acs.shape}")
print(f"\nColumns:\n{list(acs.columns)}")

In [ ]:
acs.head(10)

In [ ]:
acs.info()

In [ ]:
acs.describe()

In [ ]:
print("Missing values:\n", acs.isnull().sum())
print("\nDuplicates:", acs.duplicated().sum())

## 2. CDC PLACES Raw Data

Source: CDC — PLACES dataset
Purpose: Population health outcomes (diabetes, obesity, smoking, etc.)

In [ ]:
cdc = pd.read_csv(os.path.join(DATA_DIR, "cdc_raw.csv"), low_memory=False)
print(f"Shape: {cdc.shape}")
print(f"\nColumns:\n{list(cdc.columns)}")

In [ ]:
cdc.head(10)

In [ ]:
cdc.info()

In [ ]:
print("Missing values (top 10):")
missing = cdc.isnull().sum().sort_values(ascending=False)
print(missing.head(10))
print(f"\nDuplicates: {cdc.duplicated().sum()}")

In [ ]:
print("Unique categories:")
print("DataSource:", cdc["DataSource"].unique())
print("Category:", cdc["Category"].unique())
print("Year:", cdc["Year"].unique())

In [ ]:
print("Count by category:\n")
print(cdc["Category"].value_counts())

In [ ]:
print("Count by state:\n")
print(cdc["StateDesc"].value_counts().head(20))

## 3. CMS Raw Data

Source: CMS — Geographic Variation Data
Purpose: County-level Medicare utilization and spending

In [ ]:
import zipfile

# CMS file is a ZIP containing a CSV
cms_zip_path = os.path.join(DATA_DIR, "cms_raw.csv")
with zipfile.ZipFile(cms_zip_path) as z:
    csv_name = z.namelist()[0]
    with z.open(csv_name) as f:
        cms = pd.read_csv(f, low_memory=False)
    print(f"Extracted: {csv_name}")
print(f"Shape: {cms.shape}")

In [ ]:
print(f"Columns ({len(cms.columns)} total):")
for i, col in enumerate(cms.columns, 1):
    print(f"  {i}. {col}")

In [ ]:
cms.head(5)

In [ ]:
# Geographic and key columns
geo_cols = [c for c in cms.columns if any(k in c.lower() for k in ['geo', 'fips', 'county', 'state'])]
print(f"Geo columns: {geo_cols}")
print(f"\nBENE_GEO_LVL: {cms['BENE_GEO_LVL'].unique()}")
print(f"\nYears: {cms['YEAR'].unique()}")
print(f"\nSample geography:\n{cms[['BENE_GEO_LVL','BENE_GEO_DESC','BENE_GEO_CD']].drop_duplicates().head(10)}")

In [ ]:
print("Missing values (top 10):")
missing = cms.isnull().sum().sort_values(ascending=False)
print(missing.head(10))
print(f"\nDuplicates: {cms.duplicated().sum()}")

## 4. HRSA HPSA Raw Data

Source: HRSA — Health Professional Shortage Areas
Purpose: Healthcare access (primary care, mental health, dental shortages)

In [ ]:
hrsa = pd.read_csv(os.path.join(DATA_DIR, "hrsa_raw.csv"), low_memory=False)
print(f"Shape: {hrsa.shape}")
print(f"\nColumns:\n{list(hrsa.columns)}")

In [ ]:
hrsa.head(10)

In [ ]:
hrsa.info()

In [ ]:
print("Missing values (top 10):")
missing = hrsa.isnull().sum().sort_values(ascending=False)
print(missing.head(10))
print(f"\nDuplicates: {hrsa.duplicated().sum()}")

In [ ]:
print("Designation Type:", hrsa["Designation Type"].value_counts())
print("\nHPSA Discipline Class:", hrsa["HPSA Discipline Class"].value_counts())
print("\nRural Status:", hrsa["Rural Status"].value_counts())

In [ ]:
print("HPSA Status:", hrsa["HPSA Status"].value_counts())

## Join Key Analysis

Identify common keys for merging datasets.

In [ ]:
print("=== ACS FIPS Keys ===")
print(f"State codes: {acs['state'].nunique()} unique")
print(f"County codes: {acs['county'].nunique()} unique")
print(f"Sample FIPS: {acs[['state','county']].head()}")

In [ ]:
print("=== CDC FIPS Keys ===")
print(f"LocationID format sample: {cdc['LocationID'].head(10).tolist()}")
print(f"Unique LocationIDs: {cdc['LocationID'].nunique()}")

In [ ]:
print("=== CMS FIPS Keys ===")
cms_fips_cols = [c for c in cms.columns if 'fips' in c.lower() or 'state' in c.lower() or 'county' in c.lower()]
print(f"FIPS-related columns: {cms_fips_cols}")
print(cms[cms_fips_cols].head(5))

In [ ]:
print("=== HRSA FIPS Keys ===")
hrsa_fips_cols = [c for c in hrsa.columns if 'fips' in c.lower() or 'state' in c.lower() or 'county' in c.lower()]
print(f"FIPS-related columns: {hrsa_fips_cols}")
print(hrsa[hrsa_fips_cols].head(5))

## Summary Notes

Add observations and decisions here as you go.